# 🦈 Park Shark × 🪰 Fly Brain v2 — the fly picks the lot AND flies the route

**Run the organizers' `fly_brain_starter.ipynb` first** (we reuse its `simulate`, `rate`, `sugar`, `bitter`, `looming`, `mn9`, `giant_fibre`, `mdn`, `W`, `N`).

Every decision below is a 200 ms whole-brain run on the real 139k-neuron connectome:
1. **Lot choice** — each candidate lot's estimated availability/permits/fill become sugar/bitter/looming; MN9 vs MDN vs Giant Fibre decide.
2. **Route** — at every intersection of the campus graph, each next hop is tasted: progress toward the target = sugar, street congestion = bitter, the target filling up = looming. The brain picks the hop.
3. **Diversion** — if the Giant Fibre fires mid-route (target looming full), the fly bails to the next-ranked lot. 

Run several seeds: the brain's Poisson input noise makes each fly an individual — real neural variance, not scripted.

In [ ]:
import json, math
from google.colab import files
up = files.upload()            # upload senses.json
SJ = json.loads(next(iter(up.values())))
DAY = 'weekday'                # 'weekday' = dramatic scenario; 'sat' = today
SC = SJ['scenarios'][DAY]; G = SJ['graph']
HOUR = SC['hour']; TRAF = G['traffic_curve'][DAY][HOUR]
nodes = {n['id']: n for n in G['nodes']}
idx = {n['id']: i for i, n in enumerate(G['nodes'])}
nbrs = {}
for e in G['edges']:
    a, b = G['nodes'][e['a']]['id'], G['nodes'][e['b']]['id']
    nbrs.setdefault(a, []).append((b, e)); nbrs.setdefault(b, []).append((a, e))
def dist(u, v):
    a, b = nodes[u], nodes[v]
    dx = (a['lon']-b['lon'])*111320*math.cos(math.radians(a['lat'])); dy = (a['lat']-b['lat'])*110540
    return math.hypot(dx, dy)
print(DAY, f'{HOUR}:00 |', len(SC['lots']), 'candidate lots |', len(G['nodes']), 'nodes')

In [ ]:
# --- decision primitive: one whole-brain run -> verdict -------------------
def brain(s_hz, b_hz, l_hz, seed=0):
    inp = [(g, hz) for g, hz in [(sugar, s_hz), (bitter, b_hz), (looming, l_hz)] if hz > 0]
    c = simulate(inp, ms=200, seed=seed)
    return rate(c, mn9), rate(c, giant_fibre), rate(c, mdn)
def verdict(eat, flee, back):
    return 'FLEE' if flee > eat else ('BACK' if back > eat else 'EAT')  # escape vs feeding drive

# --- 1. lot choice --------------------------------------------------------
import pandas as pd
rows = []
for l in SC['lots']:
    e, f, b = brain(l['sugar_hz'], l['bitter_hz'], l['looming_hz'])
    rows.append({**{k: l[k] for k in ('lot','node','walk_min','sugar_hz','bitter_hz','looming_hz')},
                 'eat_mn9_hz': round(e,1), 'flee_gf_hz': round(f,1), 'back_mdn_hz': round(b,1),
                 'verdict': verdict(e, f, b)})
df = pd.DataFrame(rows).sort_values('eat_mn9_hz', ascending=False).reset_index(drop=True)
targets = df[df.verdict == 'EAT'].node.tolist() or df.node.tolist()
print('ranked targets:', [df[df.node==t].lot.iloc[0] for t in targets[:4]])
df

In [ ]:
# --- 2+3. the journey: per-hop brain decisions, Giant-Fibre diversions ----
def hop_senses(u, v, e, target):
    prog = (dist(u, target) - dist(v, target)) / max(dist(u, target), 1)   # closing in?
    tl = next((l for l in SC['lots'] if l['node'] == target), None)
    s = 200 * max(0, min(1, .5 + .5*prog)) * (1 if v != target else 1.2)
    b = 180 * TRAF * min(e['drive_min'] / 2, 1)                             # congestion on this edge
    lm = (tl['looming_hz'] if tl else 0) * (1 if v == target else .4)
    return min(round(s), 200), min(round(b), 200), min(round(lm), 150)

def fly_journey(gate, seed):
    tq = list(targets); target = tq.pop(0); u = gate; path = []
    for step in range(14):
        cands = sorted(nbrs.get(u, []), key=lambda x: dist(x[0], target))[:3]
        if not cands: break
        best = None
        for v, e in cands:
            s, b, lm = hop_senses(u, v, e, target)
            eat, flee, back = brain(s, b, lm, seed=seed + step)
            path.append({'from': u, 'to': v, 'sugar': s, 'bitter': b, 'loom': lm,
                         'eat': round(eat,1), 'flee': round(flee,1), 'back': round(back,1),
                         'verdict': verdict(eat, flee, back), 'taken': False})
            if verdict(eat, flee, back) == 'FLEE' and v == target and tq:
                target = tq.pop(0)                      # ⚡ Giant Fibre: divert!
                path[-1]['diverted_to'] = nodes[target]['name']
            if verdict(eat, flee, back) != 'FLEE' and (best is None or eat > best[2]):
                best = (v, e, eat)
        if best is None: best = (cands[0][0], cands[0][1], 0)   # all flee: creep forward
        for p in reversed(path):
            if p['from'] == u and p['to'] == best[0]: p['taken'] = True; break
        u = best[0]
        if u == target:
            return {'seed': seed, 'gate': gate, 'final': nodes[u]['name'], 'arrived': True, 'path': path}
    return {'seed': seed, 'gate': gate, 'final': nodes[u]['name'], 'arrived': False, 'path': path}

flies = [fly_journey('gate0', s) for s in (1, 2, 3)]   # 3 individual flies
for f in flies:
    hops = [p for p in f['path'] if p['taken']]
    print(f"fly seed {f['seed']}: {len(hops)} hops -> {f['final']} ({'landed' if f['arrived'] else 'wandering'})",
          '| diversions:', sum(1 for p in f['path'] if 'diverted_to' in p))

In [ ]:
# --- baseline + lesion ----------------------------------------------------
base = max(SC['lots'], key=lambda l: l['sugar_hz'] - l['bitter_hz'] - l['looming_hz'])
print('dumb baseline picks:', base['lot'], '| fly brain picks:', df.iloc[0].lot)
import numpy as np, scipy.sparse as sp
keep = np.ones(N); keep[giant_fibre] = 0
Wl = (sp.diags(keep) @ W).tocsr()
worst = max(SC['lots'], key=lambda l: l['looming_hz'])
c = simulate([(sugar, worst['sugar_hz']), (bitter, worst['bitter_hz']), (looming, worst['looming_hz'])], ms=200)
cl = simulate([(sugar, worst['sugar_hz']), (bitter, worst['bitter_hz']), (looming, worst['looming_hz'])], ms=200, W=Wl)
print(f"lesion lab @ {worst['lot']}: intact GF {rate(c, giant_fibre):.0f} Hz -> lesioned GF {rate(cl, giant_fibre):.0f} Hz",
      '(the escape reflex is gone: the fly keeps approaching a filling lot)')

In [ ]:
# --- export for the map ---------------------------------------------------
json.dump({'day': DAY, 'hour': HOUR, 'persona': SJ['persona'], 'decisions': rows},
          open('decisions.json', 'w'))
json.dump({'day': DAY, 'hour': HOUR, 'flies': flies}, open('journey.json', 'w'))
files.download('decisions.json'); files.download('journey.json')

**Say in the demo:** occupancy inputs are labeled estimates (permit-mix baseline + destination demand — no sensors yet); the brain is fixed wiring, so we show a dumb baseline and a lesion control; ~100 whole-brain runs power one journey. **Next:** Park Shark floor sensors + live Popular-Times/traffic make the senses measured instead of modeled.